In [10]:
%pip install aurite==0.3.15

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass

try:
  from google.colab import userdata #type: ignore
  # Fetch the secret value from Colab's secret manager and set it as an environment variable so your script can find it
  os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
  os.environ["SMITHERY_API_KEY"] = userdata.get('SMITHERY_API_KEY')
  os.environ["SMITHERY_PROFILE_ID"] = userdata.get('SMITHERY_PROFILE_ID')
except ImportError: # This is not a real error, just a check if running in Colab
    from dotenv import load_dotenv
    load_dotenv()  # Load environment variables from a .env file if it exists
except Exception as e:
    print(f"Error fetching from Colab's secret manager: {e}")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
if "SMITHERY_API_KEY" not in os.environ:
    os.environ["SMITHERY_API_KEY"] = getpass("Enter your Smithery API key: ")
if "SMITHERY_PROFILE_ID" not in os.environ:
    os.environ["SMITHERY_PROFILE_ID"] = getpass("Enter your Smithery Profile ID: ")

In [3]:
from IPython.display import display, Markdown

def display_agent_response(agent_name: str, query: str, response: str):
  """Formats and displays the agent's response in a structured Markdown block."""

  output = f"""
  <div style=\"border: 1px solid #D1D5DB; border-radius: 8px; margin-top: 20px; font-family: sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.05);\">
    <div style=\"background-color: #F3F4F6; padding: 10px 15px; border-bottom: 1px solid #D1D5DB; border-radius: 8px 8px 0 0;\">
      <h3 style=\"margin: 0; font-size: 16px; color: #1F2937; display: flex; align-items: center;\">
        <span style=\"margin-right: 8px;\">🤖</span>
        Agent Response: <code style=\"background-color: #E5E7EB; color: #374151; padding: 2px 6px; border-radius: 4px; margin-left: 8px;\">{agent_name}</code>
      </h3>
    </div>
    <div style=\"padding: 15px;\">
      <p style=\"margin: 0 0 10px 0; color: #6B7280; font-size: 14px;\">
        <strong>Your Query:</strong>
      </p>
      <p style=\"background-color: #F9FAFB; margin: 0 0 15px 0; color: #1F2937; border: 1px solid #E5E7EB; border-left: 3px solid #9CA3AF; padding: 10px 12px; border-radius: 4px;\">
        <em>\"{query}\"</em>
      </p>
      <hr style=\"border: none; border-top: 1px dashed #D1D5DB; margin-bottom: 15px;\">
      <p style=\"margin: 0 0 10px 0; color: #6B7280; font-size: 14px;\">
        <strong>Result:</strong>
      </p>
      <div style=\"background-color: #FFFFFF; padding: 15px; border-radius: 5px; border: 1px solid #E5E7EB; color: #1F2937; font-size: 15px; line-height: 1.6;\">
        {response}
      </div>
    </div>
  </div>
  """
  display(Markdown(output))

In [18]:
import json
import requests

def load_preferences():
    with open("preferences.json", "r") as f:
        return json.load(f)

# 2. city
def geocode(city: str):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    resp = requests.get(url, params={"name": city, "count": 1}, timeout=10)
    resp.raise_for_status()
    results = resp.json().get("results") or []
    if not results:
        raise ValueError(f"No geocoding result for city '{city}'")
    loc = results[0]
    return loc["latitude"], loc["longitude"]

# start date and end date
def get_daily_forecast(lat: float, lon: float, start: str, end: str):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "auto"
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()

def main():
    prefs = load_preferences()
    city = prefs["region"]
    start_date = prefs["start_date"]
    end_date = prefs["end_date"]

    lat, lon = geocode(city)
    data = get_daily_forecast(lat, lon, start_date, end_date)

    print(f"Weather forecast for {city}: {start_date} to {end_date}")
    for date, tmax, tmin, precip in zip(
        data["daily"]["time"],
        data["daily"]["temperature_2m_max"],
        data["daily"]["temperature_2m_min"],
        data["daily"]["precipitation_sum"]
    ):
        print(f"{date}: High {tmax}°C, Low {tmin}°C, Precip {precip} mm")

if __name__ == "__main__":
    main()

Weather forecast for Kyoto: 2025-07-13 to 2025-07-16
2025-07-13: High 33.0°C, Low 24.1°C, Precip 1.0 mm
2025-07-14: High 34.0°C, Low 25.1°C, Precip 0.8 mm
2025-07-15: High 33.9°C, Low 25.2°C, Precip 4.5 mm
2025-07-16: High 32.2°C, Low 24.5°C, Precip 3.9 mm
